# 05 · Standard CUB70 CBM: observational test of context-dependent concepts

**Report question.** On real bird photographs, do raw concept scores depend
on the visibility of the named region and on species context after exact
concept identity is held fixed?

**Causal boundary.** CUB has no accepted clean donor-part replacement.
Therefore this notebook cannot reproduce the FunnyBird donor/source
backwash predicate. It tests converging or contrary observational evidence:
natural visibility, hidden-context scores, matched recall/raw-score gaps,
and within-concept species effects.

**Population.** Standard non-RL CUB70 CBM, seed 1, epoch 100. Full-CUB CBM
is used only as a clearly labelled same-image robustness guard.


## The implemented CBM and the notation used below

For image `i`, the encoder produces one latent value for every concept slot.
The learned concept head then turns that latent value into a raw concept logit:

```text
x_i → image encoder → h_i = (h_i1, …, h_iJ)
                          ├→ learned head q_j(h_ij) → z_ij → sigmoid → p_ij
                          └→ class head on complete h_i       → species prediction
```

The implementation trains with

`L_CBM = L_task + beta × L_concept`.

The class head reads the complete latent vector `h_i`; it does not read a list of
hard 0/1 concept decisions. In these runs each concept head is a learned
`1 → 3 → 1` network, not the identity. The setup cell replays the saved head
weights on saved `h_i` and verifies that `sigmoid(z_ij)` exactly reproduces
the saved probability.

| Symbol | Meaning |
|---|---|
| `x_i` | image `i` |
| `y_i` | species label |
| `c_ij` | processed 0/1 label for exact concept `j` |
| `h_ij` | encoder's latent slot for concept `j`; also read by the class head |
| `z_ij = q_j(h_ij)` | raw concept logit after the learned head; primary grounding quantity |
| `p_ij = sigmoid(z_ij)` | bounded probability; used only for thresholded performance |
| `c_hat_ij = 1[z_ij>0]` | predicted concept presence |
| `v_ig` | whether mapped part mask `g` is visible |
| `a_ig` | visible area of mask `g` |

Ordinary accuracy and recall answer whether predictions agree with labels. They
do **not** answer whether the prediction came from the named pixels.


In [ ]:
import os, sys, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CURATED=Path(os.environ["CURATED_DATA"]); CWD=Path.cwd()
REPO=CWD if (CWD/"analysis").is_dir() else CWD.parent
sys.path.insert(0,str(REPO/"data"/"cub70"))
from cub70_parts import CUB70_PARTS, ATTRIBUTE_TYPE_TO_MASK
from relabel_cub_with_cub70 import coarse_visibility
COLORS={"head":"#56B4E9","eye":"#CC79A7","beak":"#E69F00","neck":"#009E73",
        "body":"#0072B2","wing":"#D55E00","leg":"#777777","tail":"#F0E442"}
COARSE_ORDER=["head","eye","beak","neck","body","wing","leg","tail"]
COLLAPSE_TOL=1e-8

def require(path,command):
    path=Path(path)
    if not path.exists(): raise FileNotFoundError(f"Missing {path}\nProduce it with: {command}")
    return path
def family(name): return str(name).split("::",1)[0]
def add_mapping(E):
    E=E.copy(); E["attribute_type"]=E.concept_name.map(family)
    E["mask_group"]=E.attribute_type.map(ATTRIBUTE_TYPE_TO_MASK); return E
def attach(E,V):
    local=add_mapping(E); V=V.rename(columns={"image_name":"image","coarse":"mask_group"})
    return local[local.mask_group.notna()].merge(
        V[["image","mask_group","pixel_count","area_frac","visible"]],
        on=["image","mask_group"],how="inner",validate="many_to_one")
def balanced_accuracy(y,pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

VIS=require(CURATED/"cub70_visibility.parquet","bash data/cub70/prepare_all.sh")
E70P=require(CURATED/"cub70_eval"/"cub70-cbm-s1.parquet","CONFIGS='cub70-cbm' SEEDS='1' bash analysis/cub70_prepare_analysis.sh")
EFULLP=require(CURATED/"cub70_eval"/"cub-cbm-s1.parquet","CONFIGS='cub-cbm' SEEDS='1' bash analysis/cub70_prepare_analysis.sh")
RAWVIS=pd.read_parquet(VIS); V=coarse_visibility(RAWVIS,threshold=.001)
E70=add_mapping(pd.read_parquet(E70P)); EFULL=add_mapping(pd.read_parquet(EFULLP))
J70=attach(E70,V); JFULL=attach(EFULL,V)
identity_error=float(np.nanmax(np.abs(E70.prob.to_numpy()-1/(1+np.exp(-E70.z.clip(-50,50).to_numpy())))))
if identity_error>1e-5: raise RuntimeError(f"exported z is not the concept logit: max probability mismatch={identity_error}")
print(f"[EXPORTED RAW-LOGIT PASS] max |prob-sigmoid(z)|={identity_error:.3g}")
print("CUB70 rows:",len(E70),"images:",E70.image.nunique(),"species:",E70.y_true.nunique(),"concepts:",E70.concept_name.nunique())
print("mask-matched images:",J70.image.nunique(),"fine masks:",sorted(RAWVIS.part.unique()))


## 1 · What population and mask evidence are available?

**Question.** What population and mask evidence are available?

**Variables and prediction.** Count prediction images, mask-matched images, species, exact concepts, 11 released masks, and eight coarse groups. Coverage losses must be explicit before any visible-versus-hidden comparison.

**Method.** Report fine-mask visibility and bilateral left/right support without inventing left/right concepts.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: CUB70 inventory with visibility rates and median area for all 11 released part masks.
inventory=pd.DataFrame([
    {"population":"CUB70 prediction export","images":E70.image.nunique(),"species":E70.y_true.nunique(),"concepts":E70.concept_name.nunique()},
    {"population":"mask-matched CUB70","images":J70.image.nunique(),"species":J70.y_true.nunique(),"concepts":J70.concept_name.nunique()},
])
fine=RAWVIS.groupby("part").agg(images=("image_name","nunique"),visible_rate=("visible","mean"),median_area=("area_frac","median")).reindex(CUB70_PARTS)
display(inventory); display(fine.round(4))
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
axes[0].bar(fine.index,fine.visible_rate,color="#0072B2"); axes[0].tick_params(axis="x",rotation=55)
axes[0].set_ylabel("fraction of images with visible mask"); axes[0].set_title("A · Visibility of all 11 released masks")
axes[1].bar(fine.index,fine.median_area,color="#E69F00"); axes[1].tick_params(axis="x",rotation=55)
axes[1].set_ylabel("median mask area / image area"); axes[1].set_title("B · Visible-region size")
fig.suptitle("Figure 1 · CUB70 mask population and coverage")
plt.tight_layout(); plt.show()


### Review record for Figure 1

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 2 · Is species–concept structure available before model behavior?

**Question.** Is species–concept structure available before model behavior?

**Variables and prediction.** For each exact selected concept, count supporting species, positive images, and the number of alternatives in its attribute type. Uneven support and species association make contextual prediction possible but do not prove model use.

**Method.** Use labels only; no model score appears in this figure.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Three aligned CUB70 label-only dot plots showing species support, positive-image support, and number of alternatives for every exact concept; gray marks attribute types without a released-mask mapping.
LABEL=(E70.groupby(["attribute_type","concept_name","y_true"]).gt_label.mean().reset_index())
support=(LABEL.assign(supports=lambda d:d.gt_label>=.5).groupby(["attribute_type","concept_name"])
         .agg(species_support=("supports","sum"),species_total=("y_true","nunique")).reset_index())
pos=E70.groupby(["attribute_type","concept_name"]).gt_label.agg(positive_images="sum",total_images="size").reset_index()
support=support.merge(pos); support["alternatives_in_type"]=support.groupby("attribute_type").concept_name.transform("nunique")
support["mask_group"]=support.attribute_type.map(ATTRIBUTE_TYPE_TO_MASK)
support=support.sort_values(["attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(support)); fig,axes=plt.subplots(1,3,figsize=(15,max(10,.18*len(support))),sharey=True)
plot_colors=support.mask_group.map(COLORS).fillna("#BBBBBB")
axes[0].scatter(support.species_support,y,c=plot_colors,s=18)
axes[1].scatter(support.positive_images,y,c="#0072B2",s=18)
axes[2].scatter(support.alternatives_in_type,y,c="#E69F00",s=18)
axes[0].set_yticks(y); axes[0].set_yticklabels(support.concept_name,fontsize=5); axes[0].invert_yaxis()
for ax,label in zip(axes,["species carrying exact value","positive images","values in attribute type"]): ax.set_xlabel(label)
from matplotlib.lines import Line2D
shown=[g for g in COARSE_ORDER if (support.mask_group==g).any()]
handles=[Line2D([0],[0],marker="o",linestyle="",color=COLORS[g],label=g) for g in shown]
if support.mask_group.isna().any():
    handles.append(Line2D([0],[0],marker="o",linestyle="",color="#BBBBBB",label="no released-mask mapping"))
axes[2].legend(handles=handles,loc="upper left",bbox_to_anchor=(1.02,1),fontsize=7,title="mask link")
fig.suptitle("Figure 2 · Exact-concept structure before model behavior")
plt.tight_layout(); plt.show(); display(support.round(3))


### Review record for Figure 2

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 2b · How much species identity is recoverable from the learned CUB70 concept vector?

**Question.** How much species identity is recoverable from the learned CUB70 concept vector?

**Variables and prediction.** Decode species from all raw concept logits and from each coarse mask-linked block on a held-out split. Accuracy above the 1/70 chance level shows that the learned representation stores species information; it does not prove that species caused a particular concept score.

**Method.** Build one image-by-concept matrix and use a fixed stratified 70/30 split.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Held-out CUB70 species-decoding accuracy from the complete raw concept vector and each coarse mask-linked concept block.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
X=E70.pivot_table(index="image",columns="concept_name",values="z",aggfunc="first")
y=E70[["image","y_true"]].drop_duplicates().set_index("image").loc[X.index,"y_true"]
tr,te=train_test_split(np.arange(len(X)),test_size=.30,random_state=20260803,stratify=y)
cmap=E70[["concept_name","mask_group"]].drop_duplicates().set_index("concept_name").mask_group
blocks={"complete z":list(X.columns)}
blocks.update({g:[c for c in X.columns if cmap.get(c)==g] for g in COARSE_ORDER})
rows=[]
for name,cols in blocks.items():
    if not cols: continue
    model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    model.fit(X.iloc[tr][cols],y.iloc[tr]); rows.append({"block":name,"species_accuracy":accuracy_score(y.iloc[te],model.predict(X.iloc[te][cols])),"dimensions":len(cols)})
SPECIES_PROBE=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(9,4)); ax.bar(SPECIES_PROBE.block,SPECIES_PROBE.species_accuracy,color=["#333333"]+[COLORS.get(x,"#BBBBBB") for x in SPECIES_PROBE.block.iloc[1:]])
ax.axhline(1/y.nunique(),color="black",ls="--",label="chance = 1/70"); ax.set_ylim(0,1)
ax.set_ylabel("held-out species accuracy"); ax.set_title("Figure 2b · Species decoded from CUB70 raw concept logits")
ax.legend(); plt.tight_layout(); plt.show(); display(SPECIES_PROBE.round(3))


### Review record for Figure 2b

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 3 · Did the standard CUB70 CBM produce usable exact-concept outputs?

**Question.** Did the standard CUB70 CBM produce usable exact-concept outputs?

**Variables and prediction.** For every concept, compute raw-score spread, label separation, balanced accuracy, and positive recall. Exact collapse means `Q95(z)-Q05(z) <= 1e-8`; rounded probabilities are not used to diagnose collapse.

**Method.** Evaluate all 112 outputs and mark mask-testable concepts separately.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Four aligned raw-score and thresholded-health plots for every CUB70 exact concept, with exact collapsed slots reported.
rows=[]
for (t,c),d in E70.groupby(["attribute_type","concept_name"]):
    pos=d[d.gt_label==1].z; neg=d[d.gt_label==0].z
    spread=np.quantile(d.z,.95)-np.quantile(d.z,.05)
    rows.append({"attribute_type":t,"concept_name":c,"mask_group":d.mask_group.iloc[0],
                 "spread":spread,"collapsed":spread<=COLLAPSE_TOL,
                 "label_separation":pos.median()-neg.median() if len(pos) and len(neg) else np.nan,
                 "balanced_accuracy":balanced_accuracy(d.gt_label,d.z>0),
                 "positive_recall":((pos>0).mean() if len(pos) else np.nan),
                 "n_positive":len(pos),"n_negative":len(neg)})
HEALTH=pd.DataFrame(rows).sort_values(["attribute_type","concept_name"]).reset_index(drop=True)
images=E70[["image","y_true","y_pred"]].drop_duplicates("image")
display(pd.DataFrame([{"images":len(images),"species":images.y_true.nunique(),
                      "task_accuracy":(images.y_true==images.y_pred).mean(),
                      "concept_accuracy":(E70.gt_label==E70.pred_label).mean()}]).round(4))
y=np.arange(len(HEALTH)); metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(16,max(12,.18*len(HEALTH))),sharey=True)
colors=HEALTH.mask_group.map(COLORS).fillna("#BBBBBB")
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=colors,s=17); ax.set_xlabel(m.replace("_"," "))
    if m=="label_separation": ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept_name,fontsize=5); axes[0].invert_yaxis()
fig.suptitle("Figure 3 · Raw-score health guard for every exact CUB70 concept")
plt.tight_layout(); plt.show(); display(HEALTH[HEALTH.collapsed])
print("exact collapsed slots:",int(HEALTH.collapsed.sum()),"tolerance:",COLLAPSE_TOL)


### Review record for Figure 3

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 4 · How often is a positive label paired with no visible mapped region?

**Question.** How often is a positive label paired with no visible mapped region?

**Variables and prediction.** For concept `j`, conflict is `P(v_ig=0 | c_ij=1)`. High conflict means training/evaluation labels can be predicted without visible named-region evidence; it does not prove model use.

**Method.** Plot every exact mask-testable concept at a named y-position with its denominator.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Aligned named dot plot of positive-label/mask conflict rates and denominators for every testable CUB70 exact concept.
exact=[]
for (t,c),d in J70.groupby(["attribute_type","concept_name"]):
    pos=d[d.gt_label==1]; vis=pos[pos.visible]; hid=pos[~pos.visible]
    neg_hid=d[(d.gt_label==0)&(~d.visible)]
    exact.append({"attribute_type":t,"concept_name":c,"mask_group":d.mask_group.iloc[0],
                  "n_positive":len(pos),"n_visible":len(vis),"n_hidden":len(hid),
                  "label_mask_conflict":len(hid)/len(pos) if len(pos) else np.nan,
                  "z_visible":vis.z.mean() if len(vis) else np.nan,
                  "z_hidden":hid.z.mean() if len(hid) else np.nan,
                  "visibility_effect":vis.z.mean()-hid.z.mean() if len(vis) and len(hid) else np.nan,
                  "context_gap":hid.z.mean()-neg_hid.z.mean() if len(hid) and len(neg_hid) else np.nan,
                  "n_hidden_negative":len(neg_hid)})
# `support` carries a plotting-only mask_group column. Keep the
# row-level mask_group above instead of creating mask_group_x/y.
EXACT=pd.DataFrame(exact).merge(
    support.drop(columns=["mask_group"],errors="ignore"),
    on=["attribute_type","concept_name"],how="left"
)
EXACT=EXACT.sort_values(["attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(EXACT)); fig,ax=plt.subplots(figsize=(11,max(10,.20*len(EXACT))))
ax.scatter(EXACT.label_mask_conflict,y,c=EXACT.mask_group.map(COLORS).fillna("#BBBBBB"),s=24)
ax.set_yticks(y); ax.set_yticklabels(EXACT.concept_name,fontsize=5); ax.invert_yaxis()
ax.set_xlim(-.02,1.02); ax.set_xlabel("fraction of positive labels with mapped mask absent")
ax.set_title("Figure 4 · Label/mask conflict for every exact testable concept")
plt.tight_layout(); plt.show(); display(EXACT[["concept_name","mask_group","n_positive","n_hidden","label_mask_conflict"]].round(3))


### Review record for Figure 4

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 5 · Does natural visibility change the raw score of a positive-labelled concept?

**Question.** Does natural visibility change the raw score of a positive-labelled concept?

**Variables and prediction.** `visibility_effect_j = mean(z|c=1,v=1)-mean(z|c=1,v=0)`. Positive values mean visible examples score higher; negative values require investigation rather than automatic backwash language.

**Method.** Require at least ten visible and ten hidden positive examples and show every eligible exact concept.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Zero-centered raw-logit visibility effects for every eligible CUB70 exact concept with visible and hidden counts.
VE=EXACT[(EXACT.n_visible>=10)&(EXACT.n_hidden>=10)&EXACT.visibility_effect.notna()].copy()
VE=VE.sort_values(["mask_group","attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(VE)); fig,ax=plt.subplots(figsize=(11,max(9,.23*len(VE))))
ax.scatter(VE.visibility_effect,y,c=VE.mask_group.map(COLORS).fillna("#BBBBBB"),s=30)
ax.axvline(0,color="black",lw=1); ax.set_yticks(y); ax.set_yticklabels(VE.concept_name,fontsize=6); ax.invert_yaxis()
ax.set_xlabel("visibility_effect in raw z units (visible − hidden)")
ax.set_title("Figure 5 · Natural-visibility effect for every eligible exact concept")
plt.tight_layout(); plt.show(); display(VE[["concept_name","mask_group","n_visible","n_hidden","z_hidden","z_visible","visibility_effect"]].round(3))


### Review record for Figure 5

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 6 · Does contextual concept information remain when the named region is hidden?

**Question.** Does contextual concept information remain when the named region is hidden?

**Variables and prediction.** `context_gap_j = mean(z|c=1,v=0)-mean(z|c=0,v=0)`. A positive gap means outside-region information distinguishes the label while the mapped region is hidden; it is not a donor/source margin.

**Method.** Require at least ten hidden positives and ten hidden negatives.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Zero-centered raw-logit hidden-context gaps for every eligible CUB70 exact concept.
CG=EXACT[(EXACT.n_hidden>=10)&(EXACT.n_hidden_negative>=10)&EXACT.context_gap.notna()].copy()
CG=CG.sort_values(["mask_group","attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(CG)); fig,ax=plt.subplots(figsize=(11,max(9,.23*len(CG))))
ax.scatter(CG.context_gap,y,c=CG.mask_group.map(COLORS).fillna("#BBBBBB"),s=30)
ax.axvline(0,color="black",lw=1); ax.set_yticks(y); ax.set_yticklabels(CG.concept_name,fontsize=6); ax.invert_yaxis()
ax.set_xlabel("context_gap in raw z units (hidden positive − hidden negative)")
ax.set_title("Figure 6 · Hidden-region contextual separation")
plt.tight_layout(); plt.show(); display(CG[["concept_name","mask_group","n_hidden","n_hidden_negative","context_gap"]].round(3))


### Review record for Figure 6

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 7 · Do bilateral visibility and visible area offer simpler explanations?

**Question.** Do bilateral visibility and visible area offer simpler explanations?

**Variables and prediction.** For eye, wing, and leg, retain left/right masks and compare zero, one, or two visible sides. Separately estimate within-concept area dose response. A monotone increase supports local visual evidence; non-monotone patterns motivate pose or species controls.

**Method.** Use only positive-labelled rows and raw `z`.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: CUB70 raw-logit response by number of visible bilateral masks and by within-concept visible-area quartiles.
pairmap={"eye":["left_eye","right_eye"],"wing":["left_wing","right_wing"],"leg":["left_leg","right_leg"]}
side=[]
for group,parts2 in pairmap.items():
    d=RAWVIS[RAWVIS.part.isin(parts2)]
    pv=d.pivot(index="image_name",columns="part",values="visible").fillna(False)
    pa=d.pivot(index="image_name",columns="part",values="area_frac").fillna(0)
    for image in pv.index:
        side.append({"image":image,"mask_group":group,"visible_sides":int(pv.loc[image].sum()),"bilateral_area":float(pa.loc[image].sum())})
SIDE=pd.DataFrame(side)
B=J70[(J70.gt_label==1)&J70.mask_group.isin(pairmap)].merge(SIDE,on=["image","mask_group"])
BS=B.groupby(["mask_group","visible_sides"]).agg(n=("z","size"),mean_z=("z","mean")).reset_index()
dose=[]
for (t,c),d in J70[(J70.gt_label==1)&(J70.area_frac>0)].groupby(["attribute_type","concept_name"]):
    if len(d)<20 or d.area_frac.nunique()<4: continue
    q=pd.qcut(d.area_frac,4,duplicates="drop")
    if q.nunique()<2: continue
    lo=d.loc[q==q.cat.categories[0],"z"].mean(); hi=d.loc[q==q.cat.categories[-1],"z"].mean()
    dose.append({"attribute_type":t,"concept_name":c,"mask_group":d.mask_group.iloc[0],"area_effect":hi-lo,"n":len(d)})
DOSE=pd.DataFrame(dose)
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for g,d in BS.groupby("mask_group"): axes[0].plot(d.visible_sides,d.mean_z,"o-",label=g)
axes[0].set_xticks([0,1,2]); axes[0].set_xlabel("visible left/right masks"); axes[0].set_ylabel("mean raw z"); axes[0].legend()
for g,d in DOSE.groupby("mask_group"): axes[1].scatter([g]*len(d),d.area_effect,label=g,alpha=.65)
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("largest-area quartile z − smallest-area quartile z")
fig.suptitle("Figure 7 · Bilateral visibility and area dose response")
plt.tight_layout(); plt.show(); display(BS.round(3)); display(DOSE.round(3))


### Review record for Figure 7

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 8 · Does concept performance differ between species after support is matched?

**Question.** Does concept performance differ between species after support is matched?

**Variables and prediction.** For each exact concept, compare species that each contain at least three positive and three negative images. Equalize positive sample counts and measure both recall gap and positive-row raw-z gap. Persistent gaps support species-dependent representation but remain observational.

**Method.** Use deterministic bootstrap resampling, at most 50 species pairs per exact concept, and report eligibility.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Aligned CUB70 exact-concept plots of matched per-species positive-recall gaps and raw-logit gaps.
rng=np.random.default_rng(20260803); rows=[]; B=100
for (t,c),d in E70.groupby(["attribute_type","concept_name"]):
    eligible=[]
    for sid,g in d.groupby("y_true"):
        pos=g[g.gt_label==1]; neg=g[g.gt_label==0]
        if len(pos)>=3 and len(neg)>=3: eligible.append((int(sid),pos.z.to_numpy()))
    pairs=[(eligible[a],eligible[b]) for a in range(len(eligible)) for b in range(a+1,len(eligible))]
    if len(pairs)>50: pairs=[pairs[i] for i in rng.choice(len(pairs),50,replace=False)]
    for (sa,za),(sb,zb) in pairs:
        m=min(len(za),len(zb)); rec=[]; zg=[]
        for _ in range(B):
            aa=rng.choice(za,m,replace=True); bb=rng.choice(zb,m,replace=True)
            rec.append(abs((aa>0).mean()-(bb>0).mean())); zg.append(abs(aa.mean()-bb.mean()))
        rows.append({"attribute_type":t,"concept_name":c,"species_a":sa,"species_b":sb,
                     "matched_positive_n":m,"recall_gap":np.mean(rec),"raw_z_gap":np.mean(zg)})
RECALL=pd.DataFrame(rows)
RS=(RECALL.groupby(["attribute_type","concept_name"]).agg(n_species_pairs=("recall_gap","size"),
     mean_recall_gap=("recall_gap","mean"),mean_raw_z_gap=("raw_z_gap","mean")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(13,max(8,.20*len(RS))),sharey=True)
RS=RS.sort_values(["attribute_type","concept_name"]).reset_index(drop=True); y=np.arange(len(RS))
axes[0].scatter(RS.mean_recall_gap,y,c="#0072B2",s=24); axes[1].scatter(RS.mean_raw_z_gap,y,c="#E69F00",s=24)
axes[0].set_yticks(y); axes[0].set_yticklabels(RS.concept_name,fontsize=5); axes[0].invert_yaxis()
axes[0].set_xlabel("matched absolute positive-recall gap"); axes[1].set_xlabel("matched absolute positive-row raw-z gap")
fig.suptitle("Figure 8 · Species-matched concept differences")
plt.tight_layout(); plt.show(); display(RS.round(3))


### Review record for Figure 8

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 9 · Do conflict, support, and number of alternatives organize the exact-concept effects?

**Question.** Do conflict, support, and number of alternatives organize the exact-concept effects?

**Variables and prediction.** At the concept level, relate `visibility_effect` and `context_gap` to label/mask conflict, image support, species support, and alternatives in the attribute type. Held-out predictive improvement supports an organizing association, not a causal contribution.

**Method.** Use standardized numeric predictors and repeated five-fold ridge regression.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Cross-validated concept-level error after sequentially adding label conflict, image support, species support, and number of alternatives.
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
FEATURES=["label_mask_conflict","n_positive","species_support","alternatives_in_type"]
rows=[]
for outcome in ["visibility_effect","context_gap"]:
    d=EXACT.dropna(subset=[outcome]).copy()
    cv=RepeatedKFold(n_splits=5,n_repeats=10,random_state=20260803)
    baseline=np.sqrt(np.mean((d[outcome]-d[outcome].mean())**2))
    for k in range(1,len(FEATURES)+1):
        model=make_pipeline(SimpleImputer(),StandardScaler(),Ridge(alpha=5.0))
        mse=-cross_val_score(model,d[FEATURES[:k]],d[outcome],cv=cv,scoring="neg_mean_squared_error")
        rows.append({"outcome":outcome,"stage":" + ".join(FEATURES[:k]),"rmse":float(np.sqrt(mse.mean())),"n_concepts":len(d)})
    rows.append({"outcome":outcome,"stage":"intercept only","rmse":baseline,"n_concepts":len(d)})
CONCEPT_ACCOUNT=pd.DataFrame(rows)
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for ax,outcome in zip(axes,["visibility_effect","context_gap"]):
    d=CONCEPT_ACCOUNT[CONCEPT_ACCOUNT.outcome==outcome]
    order=["intercept only"]+[" + ".join(FEATURES[:k]) for k in range(1,len(FEATURES)+1)]
    d=d.set_index("stage").reindex(order); ax.plot(range(len(d)),d.rmse,"o-")
    ax.set_xticks(range(len(d))); ax.set_xticklabels(["baseline","+ conflict","+ image support","+ species support","+ alternatives"],rotation=25,ha="right")
    ax.set_ylabel("cross-validated RMSE"); ax.set_title(outcome.replace("_"," "))
fig.suptitle("Figure 9 · Concept-level sequential observational accounting")
plt.tight_layout(); plt.show(); display(CONCEPT_ACCOUNT.round(3))


### Review record for Figure 9

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 10 · Does species explain raw-score variation within the same exact concept and visibility state?

**Question.** Does species explain raw-score variation within the same exact concept and visibility state?

**Variables and prediction.** First center `z` within each exact concept and visibility state, then summarize residual means by species. Persistent spread shows species-dependent contextual prediction beyond the current mask state.

**Method.** Require at least three rows for every displayed concept/state/species estimate.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: CUB70 species-level raw-logit residuals after centering within exact concept and visibility state for all eight coarse groups.
R=J70.copy(); R["concept_visibility_mean"]=R.groupby(["concept_name","visible"]).z.transform("mean")
R["z_after_concept_visibility"]=R.z-R.concept_visibility_mean
SP=(R.groupby(["mask_group","concept_name","visible","y_true"]).agg(n=("z","size"),residual=("z_after_concept_visibility","mean"))
      .reset_index().query("n>=3"))
fig,axes=plt.subplots(2,4,figsize=(16,8),sharey=True); axes=axes.ravel()
for ax,g in zip(axes,COARSE_ORDER):
    d=SP[SP.mask_group==g].sort_values("residual")
    ax.scatter(np.arange(len(d)),d.residual,s=12,color=COLORS[g],alpha=.7)
    ax.axhline(0,color="black",lw=.8); ax.set_title(f"{g}: {len(d)} estimates"); ax.set_xlabel("concept/state/species, sorted")
axes[0].set_ylabel("mean raw-z residual"); axes[4].set_ylabel("mean raw-z residual")
fig.suptitle("Figure 10 · Species variation after exact concept and mask state")
plt.tight_layout(); plt.show(); display(SP.groupby("mask_group").residual.agg(["min","median","max","std","count"]).round(3))


### Review record for Figure 10

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 11 · What remains after row-level visibility and species are added sequentially?

**Question.** What remains after row-level visibility and species are added sequentially?

**Variables and prediction.** Predict raw `z` on stable held-out image folds: exact concept baseline, then mask visibility/area, then species. A reduction in held-out error shows organization by that block; remaining error is the residual, not proof of an unknown cause.

**Method.** Use training-fold shrunken group means and identical rows at every stage.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Held-out CUB70 raw-logit prediction error after sequentially adding visibility, area, and species to exact concept identity.
A=J70.copy(); A["area_bin"]=pd.qcut(A.area_frac,4,labels=False,duplicates="drop")
A["fold"]=A.image.map(lambda x:int(hashlib.sha1(str(x).encode()).hexdigest(),16)%5)
stages=[("exact concept",["concept_name"]),("+ visibility and area",["concept_name","visible","area_bin"]),
        ("+ species",["concept_name","visible","area_bin","y_true"])]
rows=[]
for stage,cols in stages:
    pred=pd.Series(index=A.index,dtype=float)
    for fold in range(5):
        tr=A[A.fold!=fold]; te=A[A.fold==fold]; prior=tr.z.mean()
        st=tr.groupby(cols).z.agg(["mean","count"]).reset_index(); st["estimate"]=(st["mean"]*st["count"]+prior*10)/(st["count"]+10)
        j=te[cols].merge(st[cols+["estimate"]],on=cols,how="left")
        pred.loc[te.index]=j.estimate.fillna(prior).to_numpy()
    rows.append({"stage":stage,"rmse":float(np.sqrt(np.mean((A.z-pred)**2))),"mae":float(np.mean(np.abs(A.z-pred)))})
ROW_ACCOUNT=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(ROW_ACCOUNT.stage,ROW_ACCOUNT.rmse,"o-",color="#0072B2")
ax.set_ylabel("held-out RMSE of raw z"); ax.set_title("Figure 11 · Row-level sequential observational accounting")
plt.tight_layout(); plt.show(); display(ROW_ACCOUNT.round(3))


### Review record for Figure 11

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 12 · Do the numerical extremes correspond to pose, coarse masks, collapse, or contextual prediction?

**Question.** Do the numerical extremes correspond to pose, coarse masks, collapse, or contextual prediction?

**Variables and prediction.** Select cases by declared numerical rules: high conflict/high context gap, high conflict/low gap, strong positive visibility effect, and negative visibility effect. The photograph and all 11 masks must be inspected before assigning an explanation.

**Method.** Display original image, complete mask overlay, exact variables, species, and sample counts.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Four rule-selected CUB70 cases, each showing hidden and visible photographs beside overlays of all available released masks and exact raw-logit records.
from PIL import Image
import matplotlib.patches as mpatches
mask_root=CURATED/"cub70"/"masks"/"AnnotationMasksPerclass"
if not mask_root.is_dir(): mask_root=CURATED/"cub70"/"masks"
image_root=CURATED/"CUB_200_2011"/"images"; image_lookup={p.stem:p for p in image_root.rglob("*.jpg")}
eligible=EXACT[(EXACT.n_visible>=10)&(EXACT.n_hidden>=10)&EXACT.context_gap.notna()&EXACT.visibility_effect.notna()].copy()
high=eligible[eligible.label_mask_conflict>=eligible.label_mask_conflict.quantile(.75)]
picks=[("high conflict + high context gap",high.nlargest(1,"context_gap").iloc[0]),
       ("high conflict + low context gap",high.nsmallest(1,"context_gap").iloc[0]),
       ("strong positive visibility effect",eligible.nlargest(1,"visibility_effect").iloc[0]),
       ("negative visibility effect",eligible.nsmallest(1,"visibility_effect").iloc[0])]
mask_colors={p:plt.cm.tab20(i/20) for i,p in enumerate(CUB70_PARTS)}
def choose(row,state):
    d=J70[(J70.concept_name==row.concept_name)&(J70.gt_label==1)]
    d=d[d.visible] if state=="visible" else d[~d.visible]
    return d.iloc[(d.z-row.z_visible).abs().argmin()] if len(d) and state=="visible" else (d.iloc[(d.z-row.z_hidden).abs().argmin()] if len(d) else None)
def overlay(stem):
    rgb=np.asarray(Image.open(image_lookup[stem]).convert("RGB")); ov=rgb.astype(float)/255
    rr=RAWVIS[RAWVIS.image_name==stem]; cid=int(rr.class_idx.iloc[0])+1; present=[]
    for p in CUB70_PARTS:
        f=mask_root/str(cid)/f"{stem}_{p}.png"
        if not f.exists(): continue
        m=np.asarray(Image.open(f).convert("L"))>0
        if m.shape!=rgb.shape[:2]: m=np.asarray(Image.fromarray(m.astype("uint8")*255).resize((rgb.shape[1],rgb.shape[0]),Image.Resampling.NEAREST))>0
        ov[m]=.4*ov[m]+.6*np.array(mask_colors[p][:3]); present.append(p)
    return rgb,ov,present
fig,axes=plt.subplots(4,4,figsize=(16,14))
for r,(label,row) in enumerate(picks):
    for c,state in [(0,"hidden"),(2,"visible")]:
        rec=choose(row,state); axes[r,c].axis("off"); axes[r,c+1].axis("off")
        if rec is None: axes[r,c].text(.5,.5,"no example",ha="center"); continue
        rgb,ov,present=overlay(rec.image); axes[r,c].imshow(rgb); axes[r,c+1].imshow(ov)
        axes[r,c].set_title(f"{label}\n{state}: {rec.image}, species {rec.y_true}\n{row.concept_name}\nz={rec.z:.3f}, area={rec.area_frac:.4f}",fontsize=8)
        axes[r,c+1].set_title("all available masks\n"+", ".join(present),fontsize=8)
fig.suptitle("Figure 12 · Rule-selected photographs and complete mask overlays")
plt.tight_layout(); plt.show()
display(pd.DataFrame([{**{"case":label},**row.to_dict()} for label,row in picks])[["case","concept_name","mask_group","label_mask_conflict","visibility_effect","context_gap","n_visible","n_hidden"]].round(3))


### Review record for Figure 12

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 12b · Do the main observational quantities depend entirely on training with only 70 species?

**Question.** Do the main observational quantities depend entirely on training with only 70 species?

**Variables and prediction.** On the same mask-matched photographs and exact concepts, compare CUB70-CBM and full-CUB-CBM visibility effects and context gaps. Agreement supports robustness to the training species population; disagreement limits transfer between the two models.

**Method.** Use identical definitions and plot only concepts measurable in both exports.

The output below is intentionally not interpreted in advance. After execution,
its review must record: literal observation → strongest alternative explanation
→ discriminating test → limited conclusion → next question.


In [ ]:
# ALT: Same-image comparison of raw-logit visibility effects and context gaps between CUB70-trained and full-CUB-trained CBMs.
def exact_effects(J):
    rows=[]
    for (t,c),d in J.groupby(["attribute_type","concept_name"]):
        pos=d[d.gt_label==1]; vis=pos[pos.visible]; hid=pos[~pos.visible]; neg=d[(d.gt_label==0)&(~d.visible)]
        rows.append({"attribute_type":t,"concept_name":c,
                     "visibility_effect":vis.z.mean()-hid.z.mean() if len(vis)>=10 and len(hid)>=10 else np.nan,
                     "context_gap":hid.z.mean()-neg.z.mean() if len(hid)>=10 and len(neg)>=10 else np.nan})
    return pd.DataFrame(rows)
F=exact_effects(JFULL); P=EXACT.merge(F,on=["attribute_type","concept_name"],suffixes=("_cub70","_full"))
fig,axes=plt.subplots(1,2,figsize=(11,5))
for ax,m in zip(axes,["visibility_effect","context_gap"]):
    d=P.dropna(subset=[m+"_cub70",m+"_full"]); ax.scatter(d[m+"_full"],d[m+"_cub70"],s=25,alpha=.65)
    lo=min(d[m+"_full"].min(),d[m+"_cub70"].min()); hi=max(d[m+"_full"].max(),d[m+"_cub70"].max())
    ax.plot([lo,hi],[lo,hi],"k--",lw=.8); ax.axhline(0,color="gray",lw=.5); ax.axvline(0,color="gray",lw=.5)
    ax.set_xlabel("full-CUB CBM "+m.replace("_"," ")); ax.set_ylabel("CUB70 CBM "+m.replace("_"," ")); ax.set_title(f"{m.replace('_',' ')} (n={len(d)})")
fig.suptitle("Figure 12b · Same-image guard: CUB70-trained versus full-CUB-trained CBM")
plt.tight_layout(); plt.show()


### Review record for Figure 12b

- **Literal observation:** _Complete only after displaying this figure in chat._
- **Strongest alternative explanation:** _Pending visual review._
- **Discriminating test:** _Pending visual review._
- **Limited conclusion:** `INCOMPLETE — figure not yet reviewed`.
- **Next question:** _Complete after the limited conclusion is fixed._


## 13 · Direct question-matched FunnyBird/CUB evidence table

Complete the final column only after Figures 1–12 and the corresponding
FunnyBird figures have been displayed and reviewed.

| Scientific question | FunnyBird operation | CUB operation | Same operation? | Allowed conclusion |
|---|---|---|---|---|
| Are outputs usable? | raw-logit health guard | raw-logit health guard | yes | comparable model health |
| Do named pixels matter? | controlled `response_delta` | natural `visibility_effect` | no | causal FunnyBird; observational CUB |
| Does context remain? | final donor-minus-source margin | hidden `context_gap` | no | exact backwash FunnyBird; contextual separation CUB |
| Does visibility contribute? | same-render target area | natural mask state/area/sides | weaker in CUB | contributor support only |
| Does exact value matter? | post-swap value confusion | natural exact-concept matching | no | related difficulty evidence |
| Does species matter? | residual after exact value pair | residual after concept and visibility | observational in both | contextual association |
| Do training labels cause part of it? | matched RLv2, notebook 03rl | no accepted CUB retraining | no | no CUB causal label claim |

### CUB causal boundary

Notebook 05 may conclude that CUB does or does not show converging
**observational ingredients** of context-dependent concept prediction. It
may not claim a CUB donor/source backwash event because no accepted donor
response exists.


## 14 · Standard-CUB evidence ledger

| Predicate or explanation | Direct measurement | Status after review |
|---|---|---|
| population and mask coverage understood | Figure 1 | `INCOMPLETE` |
| species/concept shortcut available | Figure 2 | `INCOMPLETE` |
| species information in learned representation | Figure 2b | `INCOMPLETE` |
| exact outputs usable | Figure 3 | `INCOMPLETE` |
| label/mask conflict measured | Figure 4 | `INCOMPLETE` |
| natural visibility effect | Figure 5 | `INCOMPLETE` |
| hidden context separation | Figure 6 | `INCOMPLETE` |
| bilateral/area alternatives | Figure 7 | `INCOMPLETE` |
| matched recall and raw-z species gaps | Figure 8 | `INCOMPLETE` |
| concept-level accounting | Figure 9 | `INCOMPLETE` |
| species residual | Figure 10 | `INCOMPLETE` |
| row-level accounting | Figure 11 | `INCOMPLETE` |
| visual explanations inspected | Figure 12 | `INCOMPLETE` |
| same-image full-CUB robustness guard | Figure 12b | `INCOMPLETE` |

**Next report question.** Only after this ledger is reviewed may notebook
06 ask whether CUB MCBM changes the accepted observational quantities.


# Methods appendix · CUB edit proxies not used in the main claim

These completed attempts are preserved because they delimit what CUB's
available masks can support:

1. **Reciprocal whole-part deletion:** `METHOD NOT CALIBRATED FOR
   CROSS-DATASET CAUSAL COMPARISON`. The shared edit did not reproduce the
   clean FunnyBird deletion and sometimes damaged meaningful control regions.
2. **Randomized patch masking V1/V2:** selected examples supported local
   pixel response, but the all-part calibration and wing coverage were not
   sufficient for a population-level cross-dataset claim.
3. **Beak/tail paste pilot:** `VALID TEST, NO SUPPORT FOR POSITIVE DONOR
   RESPONSE`. Therefore its negative final margins cannot be interpreted as
   retained-source backwash.

These outcomes reject the proposed edit measurements for their intended
causal use. They do not reject the observational analyses in Figures 1–12
and do not weaken the validated FunnyBird renderer swap.

Full code and artifacts remain in the repository and `CURATED_DATA`; this
report does not rerun them.


# Provenance appendix

Record after execution: Git commit, CUB70 and full-CUB checkpoint paths,
epoch, prediction exports, visibility parquet, mask archive location,
population counts, exact collapsed-slot tolerance, all exclusions, and
output hashes.
